# LeetCode 2177 — Find Three Consecutive Integers That Sum to a Given Number bằng Python

Link đề: https://leetcode.com/problems/find-three-consecutive-integers-that-sum-to-a-given-number/ · Độ khó: **Medium** · Chủ đề: Math, Simulation

> Ghi chú: trên LeetCode bài này hiện được xếp **Medium** theo danh mục; thực chất là bài toán số học rất ngắn — notebook dùng nó để luyện kỹ năng **biến bài "tìm kiếm" thành phương trình**, tìm nhị phân trên miền số lớn, và tổng quát hóa cho `k` số liên tiếp.

Skeleton LeetCode:

```python
class Solution:
    def sumOfThree(self, num: int) -> List[int]:
        ...
```

Notebook đi từ **thử mọi số bắt đầu** → **tìm nhị phân trên số bắt đầu** → **phương trình `3x + 3 = num`** → **lấy số ở giữa `num / 3`** → **tổng quát `k` số liên tiếp**.

## 1. Mục tiêu học

1. Viết **phương trình** cho "ba số nguyên liên tiếp có tổng `num`": `(x − 1) + x + (x + 1) = 3x` ⇒ `num` phải chia hết cho 3.
2. Nhớ rằng "số nguyên" bao gồm **số âm và 0** — `num = 0` có đáp án `[−1, 0, 1]`.
3. Dùng **tìm nhị phân trên đáp án** khi hàm tổng đơn điệu — kỹ năng tổng quát cho miền tới `10¹⁵`.
4. Tránh lỗi **số thực** (`num / 3`) với số rất lớn; dùng `divmod` số nguyên.
5. Tổng quát cho **`k` số liên tiếp**: `num = k·a + k(k−1)/2`.

## 2. Đề bài & ví dụ

Cho số nguyên `num`. Trả về **ba số nguyên liên tiếp** (mảng đã sắp tăng) có tổng bằng `num`. Nếu không biểu diễn được, trả về mảng rỗng.

**Ví dụ 1:** `num = 33` → `[10, 11, 12]`.

**Ví dụ 2:** `num = 4` → `[]`.

**Ràng buộc:** `0 ≤ num ≤ 10¹⁵`.

In [1]:
import random
import time
from typing import List

examples = [(33, [10, 11, 12]), (4, [])]

for num, exp in examples:
    print(f"num={num}: num % 3 = {num % 3}, kỳ vọng {exp}")

num=33: num % 3 = 0, kỳ vọng [10, 11, 12]
num=4: num % 3 = 1, kỳ vọng []


## 3. Kiến thức nền

### 3.1 `divmod` cho số nguyên lớn — luôn chính xác

In [2]:
for num in [33, 4, 0, 10**15, 3 * (10**17 + 1)]:
    print(num, "-> divmod(num, 3) =", divmod(num, 3))

33 -> divmod(num, 3) = (11, 0)
4 -> divmod(num, 3) = (1, 1)
0 -> divmod(num, 3) = (0, 0)
1000000000000000 -> divmod(num, 3) = (333333333333333, 1)
300000000000000003 -> divmod(num, 3) = (100000000000000001, 0)


### 3.2 Số thực dấu phẩy động mất chính xác khi vượt `2⁵³`

`float` có 53 bit phần định trị ⇒ mọi số nguyên `≤ 2⁵³ ≈ 9·10¹⁵` biểu diễn chính xác, nhưng số lớn hơn thì **không**. Ràng buộc của đề (`≤ 10¹⁵`) còn an toàn, nhưng thói quen dùng `num / 3` sẽ gây lỗi trong bài tương tự có số lớn hơn.

In [3]:
big = 3 * (10**17 + 1)
print("2^53 =", 2**53)
print("int(big / 3) =", int(big / 3), "| big // 3 =", big // 3, "| bằng nhau?", int(big / 3) == big // 3)

2^53 = 9007199254740992
int(big / 3) = 100000000000000000 | big // 3 = 100000000000000001 | bằng nhau? False


### 3.3 Tìm nhị phân trên đáp án

Nếu `f(x)` **tăng nghiêm ngặt** theo `x`, có thể tìm `x` với `f(x) = target` trong `O(log(miền))` bước: so `f(mid)` với `target` để bỏ một nửa miền.

In [4]:
def bsearch_equal(f, target, lo, hi):
    while lo <= hi:
        mid = (lo + hi) // 2
        v = f(mid)
        if v == target:
            return mid
        if v < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return None

print("x với 3x + 3 = 33:", bsearch_equal(lambda x: 3 * x + 3, 33, -10, 100))

x với 3x + 3 = 33: 10


### 3.4 Tổng cấp số cộng

Tổng `k` số liên tiếp bắt đầu từ `a`: `a + (a+1) + … + (a+k−1) = k·a + k(k−1)/2`.

### 3.5 Chia số âm: Python (làm tròn xuống) khác C/C++/Java (làm tròn về 0)

Python: `a // b = floor(a / b)` và `a % b` luôn cùng dấu với `b`. C/C++/Java: `a / b` cắt phần thập phân (về 0), `a % b` cùng dấu với `a`. Khác biệt chỉ xuất hiện khi `a < 0` và **không chia hết** — nhưng đủ để làm sai các công thức tổng quát (lỗi 6 ở mục 13).

In [5]:
def c_div(a, b):                    # mô phỏng phép chia số nguyên kiểu C (cắt về 0)
    q = abs(a) // abs(b)
    return q if (a >= 0) == (b > 0) else -q

def c_mod(a, b):
    return a - b * c_div(a, b)

for a, b in [(7, 3), (-7, 3), (-6, 3), (-1, 4)]:
    print(f"  a={a:>3}, b={b}: Python {a}//{b} = {a // b:>3}, {a}%{b} = {a % b} | kiểu C: {c_div(a, b):>3}, {c_mod(a, b):>3}")

  a=  7, b=3: Python 7//3 =   2, 7%3 = 1 | kiểu C:   2,   1
  a= -7, b=3: Python -7//3 =  -3, -7%3 = 2 | kiểu C:  -2,  -1
  a= -6, b=3: Python -6//3 =  -2, -6%3 = 0 | kiểu C:  -2,   0
  a= -1, b=4: Python -1//4 =  -1, -1%4 = 3 | kiểu C:   0,  -1


## 4. Ý tưởng nền tảng / trực giác

Gọi ba số là `x, x+1, x+2`. Tổng: `3x + 3 = num` ⇒ `x = (num − 3) / 3 = num/3 − 1`. Có nghiệm nguyên ⇔ `num` chia hết cho 3.

Cách nhìn gọn hơn: đặt số **ở giữa** là `y`, ba số là `y−1, y, y+1`, tổng `= 3y`. ⇒ `y = num / 3`.

Tính tay:
- `num = 33`: `33 / 3 = 11` ⇒ `[10, 11, 12]` ✓.
- `num = 4`: `4 / 3` không nguyên ⇒ `[]` ✓.
- `num = 0`: `y = 0` ⇒ `[−1, 0, 1]` — **số âm được phép** (đề nói "integers").
- `num = 3`: `[0, 1, 2]`.

**Bảng tổng** của ba số liên tiếp: `…, −3, 0, 3, 6, 9, …` — đúng các bội của 3. Mỗi bội của 3 ứng với **đúng một** bộ ba ⇒ đáp án duy nhất.

In [6]:
for y in range(-2, 5):
    print(f"  giữa y={y:>2}: {[y - 1, y, y + 1]} tổng {3 * y}")

  giữa y=-2: [-3, -2, -1] tổng -6
  giữa y=-1: [-2, -1, 0] tổng -3
  giữa y= 0: [-1, 0, 1] tổng 0
  giữa y= 1: [0, 1, 2] tổng 3
  giữa y= 2: [1, 2, 3] tổng 6
  giữa y= 3: [2, 3, 4] tổng 9
  giữa y= 4: [3, 4, 5] tổng 12


### 4.1 Hai trực giác sai thường gặp

1. *"Chỉ số dương mới tính"* — đề nói **integers**; `0 = (−1) + 0 + 1`. Nếu đề viết "positive integers" thì điều kiện thành `num % 3 == 0` **và** `num ≥ 6`.
2. *"`num // 3` là số đầu"* — không, nó là số **giữa**; số đầu là `num // 3 − 1`. Cách nhớ: tổng = (số phần tử) × (trung bình), và với số phần tử **lẻ**, trung bình chính là phần tử giữa.

Quy tắc "tổng = số phần tử × trung bình" dùng được cho **mọi** cấp số cộng — đó là nền của công thức tổng quát `k` số ở cách 5.

## 5. Cách 1 — Brute Force / baseline: thử mọi số bắt đầu

Thử `x` từ `−1` (vì `num ≥ 0` thì số nhỏ nhất `≥ −1`) tới `num` (ba số liên tiếp có tổng `num ≥ 0`… số đầu không vượt `num/3`). Chỉ dùng được với `num` nhỏ.

In [7]:
def three_brute(num):
    for x in range(-1, num // 3 + 1):
        if x + (x + 1) + (x + 2) == num:
            return [x, x + 1, x + 2]
    return []

for num, exp in examples:
    print(three_brute(num), "(kỳ vọng", exp, ")")
print("num = 0 ->", three_brute(0))

[10, 11, 12] (kỳ vọng [10, 11, 12] )
[] (kỳ vọng [] )
num = 0 -> [-1, 0, 1]


**Độ phức tạp:** `O(num)` — `10¹⁵` bước: không khả thi. **Nút thắt:** tổng `3x + 3` **tăng đều** theo `x` ⇒ không cần thử tuần tự (tìm nhị phân) — thậm chí giải thẳng phương trình.

## 6. Ý tưởng thuật toán / bước đột phá

**Phương trình:** `3y = num` với `y` là số ở giữa.
- **nếu** `num % 3 == 0` **thì** `y = num // 3`, trả `[y − 1, y, y + 1]`;
- **ngược lại** trả `[]`.

**Tính duy nhất:** `3y = num` có tối đa một nghiệm ⇒ mảng trả về là duy nhất, có thể kiểm thử bằng so sánh bằng.

**Bất biến dùng cho tìm nhị phân:** `S(x) = 3x + 3` tăng nghiêm ngặt ⇒ tìm nhị phân trên `x ∈ [−1, num]`.

### 6.1 Nhìn bằng đồng dư modulo 3

Ba số liên tiếp luôn chứa **đúng một** số thuộc mỗi lớp dư `0, 1, 2` (mod 3) ⇒ tổng `≡ 0 + 1 + 2 = 3 ≡ 0 (mod 3)`. Bảng dưới kiểm cho mọi số bắt đầu `x ∈ [−6, 6]`: tổng luôn là bội của 3, và **mọi** bội của 3 trong khoảng đều xuất hiện đúng một lần.

In [8]:
sums = {}
for x in range(-6, 7):
    trip = [x, x + 1, x + 2]
    sums[sum(trip)] = trip
    print(f"  {trip}: dư mod 3 = {[v % 3 for v in trip]}, tổng {sum(trip):>3} ≡ {sum(trip) % 3} (mod 3)")
print("  các tổng xuất hiện:", sorted(sums), "| đúng là các bội của 3 từ -15 đến 21, mỗi giá trị một lần:",
      sorted(sums) == list(range(-15, 22, 3)))

  [-6, -5, -4]: dư mod 3 = [0, 1, 2], tổng -15 ≡ 0 (mod 3)
  [-5, -4, -3]: dư mod 3 = [1, 2, 0], tổng -12 ≡ 0 (mod 3)
  [-4, -3, -2]: dư mod 3 = [2, 0, 1], tổng  -9 ≡ 0 (mod 3)
  [-3, -2, -1]: dư mod 3 = [0, 1, 2], tổng  -6 ≡ 0 (mod 3)
  [-2, -1, 0]: dư mod 3 = [1, 2, 0], tổng  -3 ≡ 0 (mod 3)
  [-1, 0, 1]: dư mod 3 = [2, 0, 1], tổng   0 ≡ 0 (mod 3)
  [0, 1, 2]: dư mod 3 = [0, 1, 2], tổng   3 ≡ 0 (mod 3)
  [1, 2, 3]: dư mod 3 = [1, 2, 0], tổng   6 ≡ 0 (mod 3)
  [2, 3, 4]: dư mod 3 = [2, 0, 1], tổng   9 ≡ 0 (mod 3)
  [3, 4, 5]: dư mod 3 = [0, 1, 2], tổng  12 ≡ 0 (mod 3)
  [4, 5, 6]: dư mod 3 = [1, 2, 0], tổng  15 ≡ 0 (mod 3)
  [5, 6, 7]: dư mod 3 = [2, 0, 1], tổng  18 ≡ 0 (mod 3)
  [6, 7, 8]: dư mod 3 = [0, 1, 2], tổng  21 ≡ 0 (mod 3)
  các tổng xuất hiện: [-15, -12, -9, -6, -3, 0, 3, 6, 9, 12, 15, 18, 21] | đúng là các bội của 3 từ -15 đến 21, mỗi giá trị một lần: True


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tìm nhị phân trên số bắt đầu `x`

In [9]:
def three_bsearch(num):
    lo, hi = -1, max(num, 1)
    while lo <= hi:
        x = (lo + hi) // 2
        s = 3 * x + 3
        if s == num:
            return [x, x + 1, x + 2]
        if s < num:
            lo = x + 1
        else:
            hi = x - 1
    return []

for num, exp in examples:
    print(three_bsearch(num), "(kỳ vọng", exp, ")")

[10, 11, 12] (kỳ vọng [10, 11, 12] )
[] (kỳ vọng [] )


**Độ phức tạp:** `O(log num)` ≈ 50 bước với `num = 10¹⁵`. Không cần biết công thức, chỉ cần tổng đơn điệu — kỹ thuật dùng lại được cho bài khó hơn (ví dụ tổng bình phương liên tiếp).

### 7.2 Cách 3 — Giải phương trình `3x + 3 = num`

In [10]:
def three_equation(num):
    if (num - 3) % 3:
        return []
    x = (num - 3) // 3
    return [x, x + 1, x + 2]

for num, exp in examples:
    print(three_equation(num), "(kỳ vọng", exp, ")")
print("num = 0 ->", three_equation(0), "(Python: (-3) // 3 = -1 đúng)")

[10, 11, 12] (kỳ vọng [10, 11, 12] )
[] (kỳ vọng [] )
num = 0 -> [-1, 0, 1] (Python: (-3) // 3 = -1 đúng)


**Lưu ý chia số âm:** khi `num = 0`, `num − 3 = −3`; Python `−3 // 3 = −1` và `−3 % 3 = 0` — đúng. Ở C++/Java, `/` và `%` **làm tròn về 0** (khác Python với số âm không chia hết) — ở đây chia hết nên vẫn đúng, nhưng cần cẩn thận khi tổng quát hóa.

### 7.3 Cách 4 — Lấy số ở giữa bằng `divmod`

In [11]:
def three_middle(num):
    y, r = divmod(num, 3)
    return [y - 1, y, y + 1] if r == 0 else []

for num, exp in examples:
    print(three_middle(num), "(kỳ vọng", exp, ")")

[10, 11, 12] (kỳ vọng [10, 11, 12] )
[] (kỳ vọng [] )


**Độ phức tạp:** `O(1)` (với số nguyên lớn: `O(số chữ số)`).

### 7.4 Cách 5 — Tổng quát: `k` số nguyên liên tiếp

`num = k·a + k(k−1)/2` ⇒ `a = (num − k(k−1)/2) / k` phải nguyên. Với `k = 3`: `a = (num − 3)/3` — trùng cách 3.

In [12]:
def k_consecutive(num, k):
    t = num - k * (k - 1) // 2
    if t % k:
        return []
    a = t // k
    return list(range(a, a + k))

def k_consecutive_brute(num, k, span=60):
    for a in range(-span, span):
        if sum(range(a, a + k)) == num:
            return list(range(a, a + k))
    return []

for k in [1, 2, 3, 4, 5]:
    print(f"k={k}: num=33 -> {k_consecutive(33, k)}, num=10 -> {k_consecutive(10, k)}")

k=1: num=33 -> [33], num=10 -> [10]
k=2: num=33 -> [16, 17], num=10 -> []
k=3: num=33 -> [10, 11, 12], num=10 -> []
k=4: num=33 -> [], num=10 -> [1, 2, 3, 4]
k=5: num=33 -> [], num=10 -> [0, 1, 2, 3, 4]


Nhận xét: với `k` **lẻ**, `k(k−1)/2` chia hết cho `k` ⇒ điều kiện là `num % k == 0`; với `k` **chẵn**, điều kiện là `num ≡ k/2 (mod k)`.

## 8. TRACE từng bước

### 8.1 Trace tìm nhị phân với `num = 33` và `num = 4`

In [13]:
def trace_bsearch(num):
    lo, hi = -1, max(num, 1); step = 0
    print(f"  num = {num}")
    while lo <= hi:
        step += 1
        x = (lo + hi) // 2; s = 3 * x + 3
        rel = "=" if s == num else ("<" if s < num else ">")
        print(f"    bước {step}: lo={lo}, hi={hi}, x={x}, tổng 3x+3={s} {rel} {num}")
        if s == num:
            print("    => tìm thấy", [x, x + 1, x + 2]); return [x, x + 1, x + 2]
        if s < num: lo = x + 1
        else: hi = x - 1
    print("    => lo > hi: không có nghiệm"); return []

_ = trace_bsearch(33)
_ = trace_bsearch(4)

  num = 33
    bước 1: lo=-1, hi=33, x=16, tổng 3x+3=51 > 33
    bước 2: lo=-1, hi=15, x=7, tổng 3x+3=24 < 33
    bước 3: lo=8, hi=15, x=11, tổng 3x+3=36 > 33
    bước 4: lo=8, hi=10, x=9, tổng 3x+3=30 < 33
    bước 5: lo=10, hi=10, x=10, tổng 3x+3=33 = 33
    => tìm thấy [10, 11, 12]
  num = 4
    bước 1: lo=-1, hi=4, x=1, tổng 3x+3=6 > 4
    bước 2: lo=-1, hi=0, x=-1, tổng 3x+3=0 < 4
    bước 3: lo=0, hi=0, x=0, tổng 3x+3=3 < 4
    => lo > hi: không có nghiệm


### 8.2 Trace công thức trên các số tự dựng (đã đối chiếu vét cạn — đối chiếu lại đề gốc)

In [14]:
print(f"  {'num':>6} | {'num % 3':>7} | {'giữa':>5} | kết quả")
for num in [0, 1, 2, 3, 6, 99, 100, 300]:
    y, r = divmod(num, 3)
    res = [y - 1, y, y + 1] if r == 0 else []
    assert res == three_brute(num)
    print(f"  {num:>6} | {r:>7} | {y if r == 0 else '-':>5} | {res}")

     num | num % 3 |  giữa | kết quả
       0 |       0 |     0 | [-1, 0, 1]
       1 |       1 |     - | []
       2 |       2 |     - | []
       3 |       0 |     1 | [0, 1, 2]
       6 |       0 |     2 | [1, 2, 3]
      99 |       0 |    33 | [32, 33, 34]
     100 |       1 |     - | []
     300 |       0 |   100 | [99, 100, 101]


### 8.3 Trace công thức tổng quát `k` số liên tiếp với `num = 33`

In [15]:
def trace_k(num, kmax):
    print(f"  num = {num}")
    for k in range(1, kmax + 1):
        base = k * (k - 1) // 2
        t = num - base
        ok = t % k == 0
        detail = f"a = {t} / {k} = {t // k} -> {list(range(t // k, t // k + k))}" if ok else "không chia hết -> không có"
        print(f"    k={k}: k(k-1)/2 = {base:>2}, num - k(k-1)/2 = {t:>3}, chia {k} dư {t % k}: {detail}")

trace_k(33, 7)

  num = 33
    k=1: k(k-1)/2 =  0, num - k(k-1)/2 =  33, chia 1 dư 0: a = 33 / 1 = 33 -> [33]
    k=2: k(k-1)/2 =  1, num - k(k-1)/2 =  32, chia 2 dư 0: a = 32 / 2 = 16 -> [16, 17]
    k=3: k(k-1)/2 =  3, num - k(k-1)/2 =  30, chia 3 dư 0: a = 30 / 3 = 10 -> [10, 11, 12]
    k=4: k(k-1)/2 =  6, num - k(k-1)/2 =  27, chia 4 dư 3: không chia hết -> không có
    k=5: k(k-1)/2 = 10, num - k(k-1)/2 =  23, chia 5 dư 3: không chia hết -> không có
    k=6: k(k-1)/2 = 15, num - k(k-1)/2 =  18, chia 6 dư 0: a = 18 / 6 = 3 -> [3, 4, 5, 6, 7, 8]
    k=7: k(k-1)/2 = 21, num - k(k-1)/2 =  12, chia 7 dư 5: không chia hết -> không có


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def sumOfThree(self, num: int) -> List[int]:
        mid, rem = divmod(num, 3)          # ba số liên tiếp có tổng = 3 × số ở giữa
        if rem:
            return []
        return [mid - 1, mid, mid + 1]

for num, exp in examples:
    assert Solution().sumOfThree(num) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**(⇒)** Nếu `num = (y−1) + y + (y+1)` với `y` nguyên thì `num = 3y` ⇒ `num % 3 == 0` và `y = num // 3` — chính là bộ ba ta trả về.

**(⇐)** Nếu `num % 3 == 0`, đặt `y = num // 3`, bộ ba `[y−1, y, y+1]` gồm ba số nguyên liên tiếp, đã sắp tăng, tổng `3y = num`.

**Không có nghiệm khi `num % 3 ≠ 0`:** mọi tổng ba số nguyên liên tiếp là bội của 3.

**Tìm nhị phân (cách 2):** `S(x) = 3x + 3` tăng nghiêm ngặt; miền `[−1, num]` chứa nghiệm nếu có (vì `num ≥ 0` ⇒ `x ≥ −1`, và `x ≤ num/3 ≤ num` khi `num ≥ 0`… với `num = 0`, `hi = 1` vẫn chứa `x = −1`). Bất biến: nghiệm (nếu có) luôn nằm trong `[lo, hi]`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi `x` | `O(num)` | `O(1)` |
| 2. Tìm nhị phân | `O(log num)` | `O(1)` |
| 3. Phương trình | `O(1)` | `O(1)` |
| 4. `divmod` số giữa | `O(1)` | `O(1)` |
| 5. Tổng quát `k` số | `O(k)` (để xuất mảng) | `O(k)` |

In [17]:
rnd = random.Random(2177)
qs = [rnd.randint(0, 10**15) for _ in range(100000)]
for name, fn in [("bsearch", three_bsearch), ("equation", three_equation), ("middle", three_middle), ("Solution", Solution().sumOfThree)]:
    t0 = time.perf_counter()
    for q in qs:
        fn(q)
    print(f"{name:<9}: 100000 truy vấn trong {(time.perf_counter() - t0) * 1000:7.1f} ms")

bsearch  : 100000 truy vấn trong   593.6 ms
equation : 100000 truy vấn trong    10.4 ms
middle   : 100000 truy vấn trong     9.5 ms
Solution : 100000 truy vấn trong    10.0 ms


### 11.1 Số bước của tìm nhị phân so với `log₂(num)`

Miền tìm kiếm `[−1, num]` có `num + 2` phần tử ⇒ tìm nhị phân cần khoảng `⌈log₂(num + 2)⌉` bước. Cell dưới đếm số bước thật (đếm cả trường hợp không có nghiệm) — với `num = 10¹⁵` chỉ khoảng 50 bước, nhưng vẫn chậm hơn công thức `O(1)` hàng chục lần (mục 11 đo được ~650 ms so với ~9 ms cho 10⁵ truy vấn).

In [18]:
from math import log2, ceil

def bsearch_steps(num):
    lo, hi = -1, max(num, 1); steps = 0
    while lo <= hi:
        steps += 1
        x = (lo + hi) // 2; s3 = 3 * x + 3
        if s3 == num: return steps
        if s3 < num: lo = x + 1
        else: hi = x - 1
    return steps

for num in [4, 33, 10**3, 10**6, 10**9, 10**12, 10**15, 10**15 - 1]:
    print(f"  num={num:>16}: {bsearch_steps(num):>2} bước, ceil(log2(num+2)) = {ceil(log2(num + 2))}")

  num=               4:  3 bước, ceil(log2(num+2)) = 3
  num=              33:  5 bước, ceil(log2(num+2)) = 6
  num=            1000: 10 bước, ceil(log2(num+2)) = 10
  num=         1000000: 20 bước, ceil(log2(num+2)) = 20
  num=      1000000000: 30 bước, ceil(log2(num+2)) = 30
  num=   1000000000000: 40 bước, ceil(log2(num+2)) = 40
  num=1000000000000000: 50 bước, ceil(log2(num+2)) = 50
  num= 999999999999999: 50 bước, ceil(log2(num+2)) = 50


## 12. Bộ test

In [19]:
ALL = {
    "bsearch": three_bsearch,
    "equation": three_equation,
    "middle": three_middle,
    "k=3": lambda n: k_consecutive(n, 3),
    "Solution": lambda n: Solution().sumOfThree(n),
}

def is_valid(num, out):
    if out == []:
        return num % 3 != 0
    return len(out) == 3 and out[1] == out[0] + 1 and out[2] == out[1] + 1 and sum(out) == num

def check(num, exp=None, brute=True):
    ref = three_brute(num) if brute else three_middle(num)
    assert is_valid(num, ref)
    for name, fn in ALL.items():
        got = fn(num)
        assert got == ref, (name, num, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for num, exp in examples:
    check(num, exp)
print("Ví dụ đề: OK")

check(0, [-1, 0, 1])                 # số âm được phép
check(1, []); check(2, [])
check(3, [0, 1, 2])
check(10**15 - 1, [333333333333332, 333333333333333, 333333333333334], brute=False)
check(10**15, [], brute=False)
print("Edge case: OK")

for num in range(0, 3000):
    check(num)
print("Mọi num ∈ [0, 3000) so với vét cạn: OK")

rnd = random.Random(2021)
for _ in range(20000):
    num = rnd.randint(0, 10**15)
    r = check(num, brute=False)
    assert is_valid(num, r)
print("20000 num ngẫu nhiên tới 1e15 (đối chiếu chéo + validator): OK")

for _ in range(500):
    num = rnd.randint(-40, 40); k = rnd.randint(1, 7)
    assert k_consecutive(num, k) == k_consecutive_brute(num, k)
print("Tổng quát k số liên tiếp (cả num âm) khớp vét cạn: OK")

Ví dụ đề: OK
Edge case: OK
Mọi num ∈ [0, 3000) so với vét cạn: OK


20000 num ngẫu nhiên tới 1e15 (đối chiếu chéo + validator): OK
Tổng quát k số liên tiếp (cả num âm) khớp vét cạn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Giả sử các số phải dương (hoặc không âm) ⇒ bỏ sót `num = 0`

In [20]:
def bug_positive_only(num):
    if num % 3 or num < 6:                     # SAI: nghĩ 1 + 2 + 3 = 6 là nhỏ nhất
        return []
    y = num // 3
    return [y - 1, y, y + 1]

for num in [0, 3]:
    print(f"num={num}: sai -> {bug_positive_only(num)} | đúng -> {Solution().sumOfThree(num)}")

num=0: sai -> [] | đúng -> [-1, 0, 1]
num=3: sai -> [] | đúng -> [0, 1, 2]


**Sửa:** đề nói "integers" — gồm cả số âm và 0.

### Lỗi 2 — Lấy `num // 3` làm số **đầu tiên** thay vì số ở giữa

In [21]:
def bug_wrong_center(num):
    if num % 3:
        return []
    x = num // 3
    return [x, x + 1, x + 2]                   # SAI: tổng = num + 3

print("33 -> sai:", bug_wrong_center(33), "tổng", sum(bug_wrong_center(33)), "| đúng:", Solution().sumOfThree(33))

33 -> sai: [11, 12, 13] tổng 36 | đúng: [10, 11, 12]


**Sửa:** `num // 3` là số **ở giữa**; số đầu là `num // 3 − 1`. Mẹo tự kiểm: luôn `assert sum(out) == num`.

### Lỗi 3 — Dùng số thực `num / 3` (sai khi số vượt `2⁵³`)

In [22]:
def bug_float(num):
    y = num / 3
    if y != int(y):
        return []
    y = int(y)
    return [y - 1, y, y + 1]

big = 3 * (10**17 + 1)                         # ngoài ràng buộc đề, minh họa thói quen nguy hiểm
print("trong ràng buộc (33):", bug_float(33), "| ngoài ràng buộc:", bug_float(big), "tổng =", sum(bug_float(big)), "≠", big)
print("đúng:", Solution().sumOfThree(big))

trong ràng buộc (33): [10, 11, 12] | ngoài ràng buộc: [99999999999999999, 100000000000000000, 100000000000000001] tổng = 300000000000000000 ≠ 300000000000000003
đúng: [100000000000000000, 100000000000000001, 100000000000000002]


**Vì sao sai:** `num / 3` trả `float`; với số lớn, phép chia làm tròn nên `int(y)` lệch vài đơn vị. Trong ràng buộc `≤ 10¹⁵` của đề thì vẫn đúng — nhưng đổi sang bài có `num ≤ 10¹⁸` sẽ hỏng. **Sửa:** `divmod` số nguyên.

### Lỗi 4 — Kiểm `num % 3 == 0` nhưng trả ba số **không** sắp tăng / sai kiểu

In [23]:
def bug_unsorted(num):
    if num % 3:
        return []
    y = num // 3
    return [y + 1, y, y - 1]                   # SAI: đề yêu cầu mảng đã sắp

print("33 -> sai:", bug_unsorted(33), "| đúng:", Solution().sumOfThree(33))

33 -> sai: [12, 11, 10] | đúng: [10, 11, 12]


**Sửa:** trả `[y − 1, y, y + 1]`. Đề nói rõ "as a sorted array".

### Lỗi 5 — Tìm nhị phân với miền bắt đầu từ 0 (bỏ sót nghiệm âm)

In [24]:
def bug_bsearch_range(num):
    lo, hi = 0, num                            # SAI: x có thể là -1
    while lo <= hi:
        x = (lo + hi) // 2; s = 3 * x + 3
        if s == num: return [x, x + 1, x + 2]
        if s < num: lo = x + 1
        else: hi = x - 1
    return []

print("0 -> sai:", bug_bsearch_range(0), "| đúng:", Solution().sumOfThree(0))

0 -> sai: [] | đúng: [-1, 0, 1]


**Sửa:** miền tìm kiếm phải chứa mọi nghiệm khả dĩ (`x ≥ −1`). Bài học chung: **xác định miền đáp án cẩn thận** trước khi tìm nhị phân.

### Lỗi 6 — Công thức tổng quát: viết `k // 2 * (k − 1)` thay vì `k * (k − 1) // 2`

In [25]:
def k_consecutive_bug_order(num, k):
    base = k // 2 * (k - 1)                     # SAI: chia nguyên TRƯỚC khi nhân -> mất phần lẻ khi k lẻ
    t = num - base
    if t % k:
        return []
    a = t // k
    return list(range(a, a + k))

for num, k in [(33, 3), (15, 5), (10, 4)]:
    print(f"num={num}, k={k}: sai -> {k_consecutive_bug_order(num, k)} | đúng -> {k_consecutive(num, k)}")
bad = sum(k_consecutive_bug_order(n, k) != k_consecutive(n, k) for n in range(-20, 21) for k in range(1, 8))
print("số cặp (num, k) sai trong [-20, 20] × [1, 7]:", bad)

num=33, k=3: sai -> [] | đúng -> [10, 11, 12]
num=15, k=5: sai -> [] | đúng -> [1, 2, 3, 4, 5]
num=10, k=4: sai -> [1, 2, 3, 4] | đúng -> [1, 2, 3, 4]
số cặp (num, k) sai trong [-20, 20] × [1, 7]: 55


**Vì sao sai:** `k(k−1)` luôn chẵn nên `k(k−1) // 2` chính xác, nhưng `k // 2` làm tròn **trước** ⇒ với `k` lẻ mất `(k−1)/2`. Với `k` chẵn thì `k // 2 * (k−1)` lại đúng — lỗi chỉ lộ ra ở một nửa số trường hợp (ví dụ `k = 4` vẫn đúng). Đây là lỗi kinh điển khi viết `k / 2 * (k - 1)` trong C/Java. **Sửa:** nhân trước, chia sau; và kiểm `sum(out) == num`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Các số liên tiếp có tổng …" ⇒ viết qua **số ở giữa** (số lẻ phần tử) hoặc số đầu + công thức cấp số cộng.
- Bài "tìm kiếm" trên hàm đơn điệu ⇒ tìm nhị phân; nếu hàm tuyến tính ⇒ giải thẳng.
- Số rất lớn ⇒ chỉ dùng phép **số nguyên** (`//`, `%`, `divmod`, `math.isqrt`).
- Đọc kỹ miền giá trị: "integers" (có âm) khác "positive integers".
- Luôn tự kiểm `sum(out) == num` trong test.
- Khi chuyển công thức sang C/C++/Java: cẩn thận `/` và `%` với số âm (làm tròn về 0), và tràn `int` — `num ≤ 10¹⁵` cần `long`.
- "Bài dễ" vẫn nên có **vét cạn trên miền nhỏ** (ở đây mọi `num < 3000`) — nó bắt được cả lỗi biên (`num = 0`) mà ví dụ của đề không có.
- Đáp án **duy nhất** ⇒ test bằng so sánh bằng; đáp án không duy nhất ⇒ cần validator.

## 15. Biến thể của bài

1. **`k` số nguyên liên tiếp** (cách 5).
2. **LeetCode 829 — Consecutive Numbers Sum:** đếm số cách viết `n` thành tổng các số **nguyên dương** liên tiếp (độ dài bất kỳ).
3. **Ba số chẵn/lẻ liên tiếp** (cách nhau 2): `3y = num` với `y` cùng tính chẵn lẻ yêu cầu.
4. **Ba số liên tiếp có tích bằng `num`:** tìm nhị phân trên `x` (hàm đơn điệu khi `x ≥ 0`) hoặc `round(cbrt(num))` rồi kiểm lân cận.
5. **LeetCode 2178 — Maximum Split of Positive Even Integers** (bài kế tiếp trong danh sách): cũng là "tổng cấp số cộng" + tham lam.
6. **"Liên tiếp" theo vị trí trong mảng** (khác nghĩa!): LeetCode 1550 — Three Consecutive Odds, hay tìm cửa sổ 3 phần tử có tổng cho trước (cửa sổ trượt) — code ở cell dưới.

In [26]:
def consecutive_numbers_sum(n):
    """LeetCode 829: số cách viết n = tổng k số nguyên dương liên tiếp (k ≥ 1)."""
    count, k = 0, 1
    while k * (k + 1) // 2 <= n:
        if (n - k * (k - 1) // 2) % k == 0:
            count += 1
        k += 1
    return count

def consecutive_numbers_sum_brute(n):
    return sum(1 for a in range(1, n + 1) for k in range(1, n + 1) if k * a + k * (k - 1) // 2 == n)

for n in range(1, 120):
    assert consecutive_numbers_sum(n) == consecutive_numbers_sum_brute(n)
print("829: n=15 ->", consecutive_numbers_sum(15), "(15, 7+8, 4+5+6, 1+2+3+4+5) | khớp vét cạn n < 120")

def three_product(num):
    """Biến thể 4: ba số nguyên không âm liên tiếp có tích = num (tìm nhị phân)."""
    lo, hi = 0, 2 * 10**6
    while lo <= hi:
        x = (lo + hi) // 2; p = x * (x + 1) * (x + 2)
        if p == num: return [x, x + 1, x + 2]
        if p < num: lo = x + 1
        else: hi = x - 1
    return []

print("tích: 120 ->", three_product(120), "| 121 ->", three_product(121), "| 999*1000*1001 ->", three_product(999 * 1000 * 1001))

829: n=15 -> 4 (15, 7+8, 4+5+6, 1+2+3+4+5) | khớp vét cạn n < 120
tích: 120 -> [4, 5, 6] | 121 -> [] | 999*1000*1001 -> [999, 1000, 1001]


In [27]:
def three_consecutive_even(num):
    # Biến thể 3: ba số chẵn liên tiếp y-2, y, y+2 (y chẵn), tổng 3y
    y, r = divmod(num, 3)
    return [y - 2, y, y + 2] if r == 0 and y % 2 == 0 else []

for num in range(-30, 31):
    brute = next(([a, a + 2, a + 4] for a in range(-40, 41, 2) if 3 * a + 6 == num), [])
    assert three_consecutive_even(num) == brute
print("ba số chẵn liên tiếp: 12 ->", three_consecutive_even(12), "| 18 ->", three_consecutive_even(18), "| 15 ->", three_consecutive_even(15))

ba số chẵn liên tiếp: 12 -> [2, 4, 6] | 18 -> [4, 6, 8] | 15 -> []


In [28]:
def three_consecutive_odds(arr):
    # LeetCode 1550 (biến thể khác nghĩa "ba số liên tiếp" = ba VỊ TRÍ liên tiếp): có 3 số lẻ đứng liền nhau?
    run = 0
    for v in arr:
        run = run + 1 if v % 2 else 0
        if run == 3:
            return True
    return False

def window3_sum_index(arr, target):
    # Vị trí đầu tiên i sao cho arr[i] + arr[i+1] + arr[i+2] == target (cửa sổ trượt cỡ 3)
    if len(arr) < 3:
        return -1
    w = sum(arr[:3])
    for i in range(len(arr) - 2):
        if i:
            w += arr[i + 2] - arr[i - 1]
        if w == target:
            return i
    return -1

rnd_v = random.Random(16)
for _ in range(500):
    a = [rnd_v.randint(-5, 9) for _ in range(rnd_v.randint(0, 12))]
    t = rnd_v.randint(-10, 20)
    assert three_consecutive_odds(a) == any(all(x % 2 for x in a[i:i + 3]) for i in range(len(a) - 2))
    assert window3_sum_index(a, t) == next((i for i in range(len(a) - 2) if sum(a[i:i + 3]) == t), -1)
print("1550 [2,6,4,1] ->", three_consecutive_odds([2, 6, 4, 1]), "| [1,2,34,3,4,5,7,23,12] ->", three_consecutive_odds([1, 2, 34, 3, 4, 5, 7, 23, 12]))
print("cửa sổ 3 có tổng 12 trong [1,5,3,4,5,2]:", window3_sum_index([1, 5, 3, 4, 5, 2], 12))

1550 [2,6,4,1] -> False | [1,2,34,3,4,5,7,23,12] -> True
cửa sổ 3 có tổng 12 trong [1,5,3,4,5,2]: 1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi `x` | vét cạn | `O(num)` | `O(1)` | kiểm thử `num` nhỏ |
| 2. Tìm nhị phân | `3x + 3` tăng | `O(log num)` | `O(1)` | khi chưa có công thức |
| 3. Phương trình | `x = (num−3)/3` | `O(1)` | `O(1)` | rõ ràng |
| 4. Số giữa | `divmod(num, 3)` | `O(1)` | `O(1)` | **nộp bài** |
| 5. `k` số | cấp số cộng | `O(k)` | `O(k)` | tổng quát hóa |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().sumOfThree(0)` và `Solution().sumOfThree(999999999999999)`?

**Bài 2 (tính tay):** Với `k = 4` số liên tiếp, những `num ∈ [0, 20]` nào biểu diễn được? Viết điều kiện tổng quát.

**Bài 3 (điền TODO):** Hoàn thiện `three_consecutive_odd(num)` — ba số **lẻ** liên tiếp (`y−2, y, y+2`) có tổng `num`, hoặc `[]`.

**Bài 4 (tự viết lại):** Viết lại cách 2 (tìm nhị phân) cho bài **tổng bình phương** ba số liên tiếp: tìm `x ≥ 0` với `x² + (x+1)² + (x+2)² = num`.

**Bài 5 (mini challenge):** Cho `num`, trả về **mọi** `k` (`1 ≤ k ≤ 100`) sao cho `num` là tổng `k` số nguyên (có thể âm) liên tiếp, kèm số đầu tiên. Kiểm bằng vét cạn với `num` nhỏ.

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def three_consecutive_odd(num):
    # TODO: num = 3y, y phải lẻ
    pass

# Bài 4 — TODO
def three_squares(num):
    # TODO: tìm nhị phân x trong [0, isqrt(num) + 1]
    pass

# Bài 5 — TODO
def all_k_representations(num, kmax=100):
    # TODO: dùng k_consecutive cho từng k
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `0 = 3 · 0`.
- Mức 2: `999999999999999 = 3 · 333333333333333`.
- Mức 3: `[−1, 0, 1]` và `[333333333333332, 333333333333333, 333333333333334]`.

**Bài 2**
- Mức 1: `num = 4a + 6`.
- Mức 2: `num ≡ 2 (mod 4)`.
- Mức 3: `2, 6, 10, 14, 18` (với `a = −1, 0, 1, 2, 3`).

**Bài 3**
- Mức 1: Tổng `3y`.
- Mức 2: `y = num // 3` phải lẻ.
- Mức 3: `num = 15` ⇒ `[3, 5, 7]`; `num = 12` ⇒ `y = 4` chẵn ⇒ `[]`.

**Bài 4**
- Mức 1: `f(x) = 3x² + 6x + 5` tăng với `x ≥ 0`.
- Mức 2: `hi = isqrt(num) + 1`.
- Mức 3: `num = 50` ⇒ `[3, 4, 5]`.

**Bài 5**
- Mức 1: Lặp `k = 1..100`, gọi `k_consecutive(num, k)`.
- Mức 2: Giữ cặp `(k, a)` khi có nghiệm.
- Mức 3: Vét cạn: thử `a` trong một khoảng đủ rộng.

## 19. Đáp án tham khảo cho bài tập

In [30]:
from math import isqrt

# Bài 1
r1 = (Solution().sumOfThree(0), Solution().sumOfThree(999999999999999))
print("Bài 1:", r1)
assert r1 == ([-1, 0, 1], [333333333333332, 333333333333333, 333333333333334])

# Bài 2
r2 = [n for n in range(21) if k_consecutive(n, 4)]
print("Bài 2:", r2, [k_consecutive(n, 4) for n in r2]); assert r2 == [2, 6, 10, 14, 18]

# Bài 3
def three_consecutive_odd_sol(num):
    y, r = divmod(num, 3)
    return [y - 2, y, y + 2] if r == 0 and y % 2 == 1 else []

for num in range(-60, 61):
    brute = next(([a, a + 2, a + 4] for a in range(-41, 41, 2) if 3 * a + 6 == num), [])
    assert three_consecutive_odd_sol(num) == brute
print("Bài 3 OK:", three_consecutive_odd_sol(15), three_consecutive_odd_sol(12), three_consecutive_odd_sol(-9))

Bài 1: ([-1, 0, 1], [333333333333332, 333333333333333, 333333333333334])
Bài 2: [2, 6, 10, 14, 18] [[-1, 0, 1, 2], [0, 1, 2, 3], [1, 2, 3, 4], [2, 3, 4, 5], [3, 4, 5, 6]]
Bài 3 OK: [3, 5, 7] [] [-5, -3, -1]


In [31]:
# Bài 4
def three_squares_sol(num):
    lo, hi = 0, isqrt(num) + 1
    while lo <= hi:
        x = (lo + hi) // 2; s = x * x + (x + 1) ** 2 + (x + 2) ** 2
        if s == num: return [x, x + 1, x + 2]
        if s < num: lo = x + 1
        else: hi = x - 1
    return []

for num in range(0, 5000):
    brute = next(([x, x + 1, x + 2] for x in range(0, 50) if x * x + (x + 1) ** 2 + (x + 2) ** 2 == num), [])
    assert three_squares_sol(num) == brute
print("Bài 4 OK:", three_squares_sol(50), three_squares_sol(51), three_squares_sol(3 * 10**12 * 10**12 + 6 * 10**12 + 5))

# Bài 5
def all_k_representations_sol(num, kmax=100):
    out = []
    for k in range(1, kmax + 1):
        r = k_consecutive(num, k)
        if r:
            out.append((k, r[0]))
    return out

def all_k_brute(num, kmax):
    return [(k, a) for k in range(1, kmax + 1) for a in range(-200, 201) if k * a + k * (k - 1) // 2 == num]

for num in range(-30, 31):
    assert all_k_representations_sol(num, 12) == all_k_brute(num, 12)
print("Bài 5 OK; num=9 ->", all_k_representations_sol(9, 10))

Bài 4 OK: [3, 4, 5] [] [1000000000000, 1000000000001, 1000000000002]
Bài 5 OK; num=9 -> [(1, 9), (2, 4), (3, 2), (6, -1), (9, -3)]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
mid, rem = divmod(num, 3)
return [mid - 1, mid, mid + 1] if rem == 0 else []
```

**Tổng quát `k` số:** `t = num − k(k−1)//2; a = t // k nếu t % k == 0`.

**Checklist:**
- [ ] Số âm và 0 được phép (`num = 0` ⇒ `[−1, 0, 1]`)?
- [ ] `num // 3` là số **ở giữa**?
- [ ] Chỉ dùng phép số nguyên (không `num / 3`)?
- [ ] Mảng trả về đã sắp tăng?
- [ ] Tự kiểm `sum(out) == num`?
- [ ] Chữ ký đúng: `def sumOfThree(self, num: int) -> List[int]`?